# Uncertainty & Calibration .


## 1. Configuration

In [ ]:
from pathlib import Path
import torch

PROJECT_ROOT = Path(r"D:\Ravishi\MSc Final Project\skin-lesion-xai")

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
UNC_DIR = RESULTS_DIR / "uncertainty"
UNC_DIR.mkdir(parents=True, exist_ok=True)

EVAL_CSV = DATA_PROCESSED / "val.csv"

IMAGE_SIZE = 224
BATCH_SIZE = 16
MC_PASSES = 20            
N_BINS = 10               
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

MODEL_CONFIGS = {
    "cnn_baseline":  {"timm_name": "resnet50",             "description": "Baseline CNN (ResNet-50)"},
    "attention_cnn": {"timm_name": "resnet50",             "description": "ResNet-50 + CBAM"},
    "vit":           {"timm_name": "deit_small_patch16_224","description": "DeiT-Small (transformer)"},
}
DISPLAY_NAMES = {"cnn_baseline": "CNN", "attention_cnn": "Attention-CNN", "vit": "DeiT"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 2. Model definitions & loading 

In [ ]:
import timm
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1); self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False),
                                 nn.ReLU(inplace=True), nn.Conv2d(hidden, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)
    def forward(self, x):
        return self.spatial_attn(self.channel_attn(x))

class AttentionCNN(nn.Module):
    def __init__(self, backbone_name="resnet50", num_classes=2, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool="")
        feat_dim = self.backbone.num_features
        self.cbam = CBAM(feat_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(feat_dim, num_classes)
    def forward(self, x):
        feats = self.cbam(self.backbone(x))
        return self.fc(self.dropout(self.pool(feats).flatten(1)))

class DropoutCNN(nn.Module):

    def __init__(self, backbone_name="resnet50", num_classes=2, pretrained=False,
                 dropout=0.5):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool="")
        feat_dim = self.backbone.num_features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        return self.fc(self.dropout(self.pool(self.backbone(x)).flatten(1)))

def build_model(model_key, num_classes=2):
    timm_name = MODEL_CONFIGS[model_key]["timm_name"]
    if model_key == "attention_cnn":
        return AttentionCNN(timm_name, num_classes, pretrained=False)
    if model_key == "cnn_baseline":
        return DropoutCNN(timm_name, num_classes, pretrained=False)
    return timm.create_model(timm_name, pretrained=False, num_classes=num_classes,
                             drop_rate=0.5)

def load_trained(model_key):
    ckpt_path = MODELS_DIR / f"{model_key}_best.pth"
    model = build_model(model_key)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt["model_state"] if "model_state" in ckpt else ckpt
    model.load_state_dict(state)
    model.to(device).eval()
    return model

print("Model definitions ready.")


## 3. Data loader

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class EvalDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self.tf = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.tf(Image.open(row["image_path"]).convert("RGB")), int(row["label"])

eval_ds = EvalDataset(EVAL_CSV)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Evaluation set: {len(eval_ds)} images")


## 4. Collect logits 


In [ ]:
def collect_logits(model):
    all_logits, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in eval_loader:
            logits = model(images.to(device)).float()
            all_logits.append(logits.cpu())
            all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)

print("Collecting logits for each model...")
logits_store = {}
for key in MODEL_CONFIGS:
    model = load_trained(key)
    lg, lb = collect_logits(model)
    logits_store[key] = {"logits": lg, "labels": lb}
    print(f"  {DISPLAY_NAMES[key]}: {lg.shape[0]} samples")
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
print("Done.")


## 5. Calibration metrics (ECE) and reliability diagrams


In [ ]:
def expected_calibration_error(probs, labels, n_bins=N_BINS):
    probs, labels = np.asarray(probs), np.asarray(labels)
    conf = np.maximum(probs, 1 - probs)            
    pred = (probs >= 0.5).astype(int)
    correct = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    total, n = 0.0, len(probs)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        total += (mask.sum() / n) * abs(correct[mask].mean() - conf[mask].mean())
    return total


def reliability_data(probs, labels, n_bins=N_BINS):
    probs, labels = np.asarray(probs), np.asarray(labels)
    conf = np.maximum(probs, 1 - probs)
    pred = (probs >= 0.5).astype(int)
    correct = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    centres, accs, confs, counts = [], [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        centres.append((lo + hi) / 2)
        if mask.sum() == 0:
            accs.append(np.nan); confs.append(np.nan); counts.append(0)
        else:
            accs.append(correct[mask].mean())
            confs.append(conf[mask].mean())
            counts.append(int(mask.sum()))
    return np.array(centres), np.array(accs), np.array(confs), np.array(counts)

print("Calibration functions ready.")


## 6. Temperature scaling


In [ ]:
import torch.nn.functional as F

def fit_temperature(logits, labels, max_iter=200):
    log_T = torch.zeros(1, requires_grad=True)   
    optimizer = torch.optim.LBFGS([log_T], lr=0.05, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        loss = F.cross_entropy(logits / torch.exp(log_T), labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    return torch.exp(log_T).item()


results = []
calibrated_probs = {}

for key in MODEL_CONFIGS:
    lg = logits_store[key]["logits"]
    lb = logits_store[key]["labels"]

    probs_before = torch.softmax(lg, dim=1)[:, 1].numpy()
    ece_before = expected_calibration_error(probs_before, lb.numpy())

    T = fit_temperature(lg, lb)
    probs_after = torch.softmax(lg / T, dim=1)[:, 1].numpy()
    ece_after = expected_calibration_error(probs_after, lb.numpy())

    calibrated_probs[key] = {"before": probs_before, "after": probs_after,
                             "labels": lb.numpy(), "T": T}
    results.append({"Model": DISPLAY_NAMES[key], "Temperature T": round(T, 3),
                    "ECE before": round(ece_before, 4), "ECE after": round(ece_after, 4)})

cal_table = pd.DataFrame(results).set_index("Model")
cal_table.to_csv(UNC_DIR / "calibration_table.csv")
print(cal_table.to_string())
print("\nT > 1 means the model was over-confident and has been softened.")
print("T < 1 means it was under-confident and has been sharpened.")
cal_table


## 7. Reliability diagrams (before vs after calibration)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, key in zip(axes, MODEL_CONFIGS):
    d = calibrated_probs[key]
    for label, probs, style in [("before", d["before"], "o-"), ("after", d["after"], "s--")]:
        centres, accs, confs, counts = reliability_data(probs, d["labels"])
        valid = ~np.isnan(accs)
        ax.plot(confs[valid], accs[valid], style, label=f"{label} (T={d['T']:.2f})")
    ax.plot([0.5, 1], [0.5, 1], "k:", alpha=0.6, label="perfect calibration")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_title(DISPLAY_NAMES[key]); ax.legend(fontsize=9)
    ax.set_xlim(0.45, 1.02); ax.set_ylim(0, 1.02)

fig.suptitle("Reliability diagrams")
plt.tight_layout()
plt.savefig(UNC_DIR / "reliability_diagrams.png", dpi=150)
plt.show()
print(f"Saved: {UNC_DIR / 'reliability_diagrams.png'}")


## 8. MC Dropout

In [ ]:
def enable_mc_dropout(model):
    model.eval()
    n_dropout = 0
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()
            n_dropout += 1
    return n_dropout


def mc_dropout_predict(model, n_passes=MC_PASSES):
    all_passes = []
    labels_out = None
    with torch.no_grad():
        for p in range(n_passes):
            pass_probs, pass_labels = [], []
            for images, labels in eval_loader:
                probs = torch.softmax(model(images.to(device)).float(), dim=1)[:, 1]
                pass_probs.append(probs.cpu().numpy())
                if p == 0:
                    pass_labels.append(np.asarray(labels))
            all_passes.append(np.concatenate(pass_probs))
            if p == 0:
                labels_out = np.concatenate(pass_labels)
    stacked = np.stack(all_passes)              
    return stacked.mean(axis=0), stacked.std(axis=0), labels_out


mc_results = {}
print(f"Running MC Dropout ({MC_PASSES} passes per model)")
for key in MODEL_CONFIGS:
    model = load_trained(key)
    n_dropout = enable_mc_dropout(model)
    mean_p, std_p, labels = mc_dropout_predict(model)
    mc_results[key] = {"mean": mean_p, "std": std_p, "labels": labels}
    print(f"  {DISPLAY_NAMES[key]}: {n_dropout} dropout layer(s) active, "
          f"mean uncertainty (std) = {std_p.mean():.4f}")
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
print("Done.")


## 9. Does uncertainty correlate with errors?

In [ ]:
rows = []
for key in MODEL_CONFIGS:
    d = mc_results[key]
    pred = (d["mean"] >= 0.5).astype(int)
    correct = pred == d["labels"]
    rows.append({
        "Model": DISPLAY_NAMES[key],
        "Uncertainty (correct)": round(float(d["std"][correct].mean()), 4),
        "Uncertainty (wrong)": round(float(d["std"][~correct].mean()), 4),
        "Ratio (wrong/correct)": round(float(d["std"][~correct].mean() /
                                            max(d["std"][correct].mean(), 1e-9)), 2),
    })

unc_table = pd.DataFrame(rows).set_index("Model")
unc_table.to_csv(UNC_DIR / "mc_dropout_uncertainty.csv")
print(unc_table.to_string())


fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, key in zip(axes, MODEL_CONFIGS):
    d = mc_results[key]
    pred = (d["mean"] >= 0.5).astype(int)
    correct = pred == d["labels"]
    ax.hist(d["std"][correct], bins=30, alpha=0.6, label="correct", density=True)
    ax.hist(d["std"][~correct], bins=30, alpha=0.6, label="wrong", density=True)
    ax.set_xlabel("MC Dropout uncertainty (std)"); ax.set_ylabel("density")
    ax.set_title(DISPLAY_NAMES[key]); ax.legend()

fig.suptitle("Predictive uncertainty: correct vs incorrect predictions")
plt.tight_layout()
plt.savefig(UNC_DIR / "uncertainty_vs_correctness.png", dpi=150)
plt.show()
print(f"\nAll outputs saved to: {UNC_DIR}")
